In [ ]:
#| default_exp models.tirex

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

# TiRex
`TiRex` is a model based on the `xLSTM` architecture. 

![Figure 1. xLSTM Architecture.](imgs_models/xlstm.png)

**References**<br>-[Auer, Andreas and Podest, Patrick and Klotz, Daniel and Bock, Sebastian and Klambauer, Gunter and Hochreiter, Sepp. "TiRex: Zero-Shot Forecasting Across Long and Short Horizons with Enhanced In-Context Learning" (2025).](https://arxiv.org/abs/2505.23719)<br>

In [ ]:
#| hide
import logging
import warnings
from fastcore.test import test_eq
from nbdev.showdoc import show_doc
from neuralforecast.common._model_checks import check_model

In [ ]:
#| export
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

from neuralforecast.losses.pytorch import MAE
from neuralforecast.common._base_model import BaseModel
from neuralforecast.common._modules import ACTIVATIONS

try:
    from xlstm.xlstm_block_stack import xLSTMBlockStack, xLSTMBlockStackConfig
    from xlstm.blocks.mlstm.block import mLSTMBlockConfig
    from xlstm.blocks.slstm.block import sLSTMBlockConfig
    IS_XLSTM_INSTALLED = True
except ImportError:
    IS_XLSTM_INSTALLED = False

In [ ]:
#| export
class ResidualBlock(nn.Module):
    """Residual Block

    **Parameters:**<br>
    `in_features`: int, dimension of input.<br>
    `out_features`: int, dimension of output.<br>
    `activation`: str, activation function to use.<br>
    `hidden_size`: int, dimension of hidden layers.<br>
    `dropout`: float, dropout rate.<br>
    """

    def __init__(
        self, in_features, out_features, activation, hidden_size, dropout
    ):
        super().__init__()
        assert activation in ACTIVATIONS, f"{activation} is not in {ACTIVATIONS}"

        self.activation = getattr(nn, activation)()
        self.dropout = nn.Dropout(dropout)
        self.hidden_layer = nn.Linear(in_features, hidden_size)
        self.output_layer = nn.Linear(hidden_size, out_features)
        self.residual_layer = nn.Linear(in_features, out_features)

    def forward(self, x):
        residual = self.residual_layer(x)
        x = self.hidden_layer(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.output_layer(x)
        return x + residual

In [ ]:
#| export
class TiRex(BaseModel):
    """ TiRex

    TiRex model

    **Parameters:**<br>
    `h`: int, forecast horizon.<br>
    `input_size`: int, considered autorregresive inputs (lags), y=[1,2,3,4] input_size=2 -> lags=[1,2].<br>
    `input_ff_dim`: int=1024, feedforward dimension for the input projection.<br>
    `input_dropout`: float=0., dropout rate for the input projection.<br>
    `input_activation`: str='ReLU', activation function for the input projection.<br>
    `input_patch_len`: int=32, length of the input patches.<br>
    `stride`: int=32, stride for the input patches.<br>
    `embedding_dim`: int=256, embedding dimension.<br>
    `xlstm_blocks`: int=1, number of xLSTM blocks.<br>
    `xlstm_backbone`: str='mLSTM', xLSTM backbone, either 'mLSTM' or 'sLSTM'.<br>
    `xlstm_bias`: bool=True, whether to use bias in xLSTM blocks.<br>
    `xlstm_dropout`: float=0.1, dropout rate for xLSTM blocks.<br>
    `xlstm_context_length`: int=2048, context length for xLSTM blocks.<br>
    `output_ff_dim`: int=1024, feedforward dimension for the output projection.<br>
    `futr_exog_list`: str list, future exogenous columns.<br>
    `hist_exog_list`: str list, historic exogenous columns.<br>
    `stat_exog_list`: str list, static exogenous columns.<br>
    `exclude_insample_y`: bool=False, whether to exclude the target variable from the input.<br>
    `recurrent`: bool=False, whether to produce forecasts recursively (True) or direct (False).<br>
    `loss`: PyTorch module, instantiated train loss class from [losses collection](https://nixtla.github.io/neuralforecast/losses.pytorch.html).<br>
    `valid_loss`: PyTorch module=`loss`, instantiated valid loss class from [losses collection](https://nixtla.github.io/neuralforecast/losses.pytorch.html).<br>
    `max_steps`: int=1000, maximum number of training steps.<br>
    `learning_rate`: float=1e-3, Learning rate between (0, 1).<br>
    `num_lr_decays`: int=-1, Number of learning rate decays, evenly distributed across max_steps.<br>
    `early_stop_patience_steps`: int=-1, Number of validation iterations before early stopping.<br>
    `val_check_steps`: int=100, Number of training steps between every validation loss check.<br>
    `batch_size`: int=32, number of differentseries in each batch.<br>
    `valid_batch_size`: int=None, number of different series in each validation and test batch.<br>
    `windows_batch_size`: int=128, number of windows to sample in each training batch, default uses all.<br>
    `inference_windows_batch_size`: int=1024, number of windows to sample in each inference batch, -1 uses all.<br>
    `start_padding_enabled`: bool=False, if True, the model will pad the time series with zeros at the beginning, by input size.<br>
    `training_data_availability_threshold`: Union[float, List[float]]=0.0, minimum fraction of valid data points required for training windows. Single float applies to both insample and outsample; list of two floats specifies [insample_fraction, outsample_fraction]. Default 0.0 allows windows with only 1 valid data point (current behavior).<br>
    `step_size`: int=1, step size between each window of temporal data.<br>    
    `scaler_type`: str='robust', type of scaler for temporal inputs normalization see [temporal scalers](https://nixtla.github.io/neuralforecast/common.scalers.html).<br>
    `random_seed`: int=1, random_seed for pytorch initializer and numpy generators.<br>
    `drop_last_loader`: bool=False, if True `TimeSeriesDataLoader` drops last non-full batch.<br>
    `alias`: str, optional,  Custom name of the model.<br>
    `optimizer`: Subclass of 'torch.optim.Optimizer', optional, user specified optimizer instead of the default choice (Adam).<br>
    `optimizer_kwargs`: dict, optional, list of parameters used by the user specified `optimizer`.<br>
    `lr_scheduler`: Subclass of 'torch.optim.lr_scheduler.LRScheduler', optional, user specified lr_scheduler instead of the default choice (StepLR).<br>
    `lr_scheduler_kwargs`: dict, optional, list of parameters used by the user specified `lr_scheduler`.<br>    
    `dataloader_kwargs`: dict, optional, list of parameters passed into the PyTorch Lightning dataloader by the `TimeSeriesDataLoader`. <br>
    `**trainer_kwargs`: int,  keyword trainer arguments inherited from [PyTorch Lighning's trainer](https://pytorch-lightning.readthedocs.io/en/stable/api/pytorch_lightning.trainer.trainer.Trainer.html?highlight=trainer).<br>

    **References:**<br>
    -[Maximilian Beck, Korbinian Pöppel, Markus Spanring, Andreas Auer, Oleksandra Prudnikova, Michael Kopp, Günter Klambauer, Johannes Brandstetter, Sepp Hochreiter (2024). "xLSTM: Extended Long Short-Term Memory"](https://arxiv.org/abs/2405.04517)

    """
    # Class attributes
    EXOGENOUS_FUTR = False
    EXOGENOUS_HIST = False
    EXOGENOUS_STAT = False
    MULTIVARIATE = False    # If the model produces multivariate forecasts (True) or univariate (False)
    RECURRENT = False        # If the model produces forecasts recursively (True) or direct (False)

    def __init__(self,
                 h: int,
                 input_size: int,
                 input_ff_dim: int = 1024,
                 input_dropout: float = 0.,
                 input_activation: str = 'ReLU',
                 input_patch_len: int = 32,
                 stride: int = 32,
                 embedding_dim: int = 256,
                 xlstm_blocks: int = 1,
                 xlstm_backbone: str = "mLSTM",
                 xlstm_bias: bool = True,
                 xlstm_dropout: float = 0.1,
                 xlstm_context_length: int = 2048,
                 output_ff_dim: int = 1024,
                 futr_exog_list = None,
                 hist_exog_list = None,
                 stat_exog_list = None,
                 exclude_insample_y = False,
                 loss = MAE(),
                 valid_loss = None,
                 max_steps: int = 1000,
                 learning_rate: float = 1e-3,
                 num_lr_decays: int = -1,
                 early_stop_patience_steps: int =-1,
                 val_check_steps: int = 100,
                 batch_size = 32,
                 valid_batch_size: Optional[int] = None,
                 windows_batch_size = 128,
                 inference_windows_batch_size = 1024,
                 start_padding_enabled = False,
                 training_data_availability_threshold = 0.0,
                 step_size: int = 1,
                 scaler_type: str = 'identity',
                 random_seed = 1,
                 drop_last_loader = False,
                 alias: Optional[str] = None,
                 optimizer = None,
                 optimizer_kwargs = None,
                 lr_scheduler = None,
                 lr_scheduler_kwargs = None,
                 dataloader_kwargs = None,
                 **trainer_kwargs):
        
        super(TiRex, self).__init__(
            h=h,
            input_size=input_size,
            futr_exog_list=futr_exog_list,
            hist_exog_list=hist_exog_list,
            stat_exog_list=stat_exog_list,
            exclude_insample_y = exclude_insample_y,
            loss=loss,
            valid_loss=valid_loss,
            max_steps=max_steps,
            learning_rate=learning_rate,
            num_lr_decays=num_lr_decays,
            early_stop_patience_steps=early_stop_patience_steps,
            val_check_steps=val_check_steps,
            batch_size=batch_size,
            valid_batch_size=valid_batch_size,
            windows_batch_size=windows_batch_size,
            inference_windows_batch_size=inference_windows_batch_size,
            start_padding_enabled=start_padding_enabled,
            training_data_availability_threshold=training_data_availability_threshold,
            step_size=step_size,
            scaler_type=scaler_type,
            random_seed=random_seed,
            drop_last_loader=drop_last_loader,
            alias=alias,
            optimizer=optimizer,
            optimizer_kwargs=optimizer_kwargs,
            lr_scheduler=lr_scheduler,
            lr_scheduler_kwargs=lr_scheduler_kwargs,
            dataloader_kwargs=dataloader_kwargs,
            **trainer_kwargs
        )

        if not IS_XLSTM_INSTALLED:
            raise ImportError(
                "Please install `xlstm`. You also need to install `mlstm_kernels` for backend='mLSTM' and `ninja` for backend='sLSTM'."
            )        

        # xLSTM input size
        n_patches = int((input_size - input_patch_len) / stride + 1) + 1
        self.stride = stride
        self.input_patch_len = min(input_size + stride, input_patch_len)            
        
        # Encoder Residual Block
        self.input_embedding = ResidualBlock(
            in_features=2 * input_patch_len,
            out_features=embedding_dim,
            activation=input_activation,
            hidden_size=input_ff_dim,
            dropout=input_dropout
        )

        if xlstm_backbone == "sLSTM":
            block_stack_config = xLSTMBlockStackConfig(slstm_block=sLSTMBlockConfig(), context_length=xlstm_context_length, num_blocks=xlstm_blocks, embedding_dim=embedding_dim, bias=xlstm_bias, dropout=xlstm_dropout)
        elif xlstm_backbone == "mLSTM":
            block_stack_config = xLSTMBlockStackConfig(mlstm_block=mLSTMBlockConfig(), context_length=xlstm_context_length, num_blocks=xlstm_blocks, embedding_dim=embedding_dim, bias=xlstm_bias, dropout=xlstm_dropout)
        self.xlstm = xLSTMBlockStack(block_stack_config)

        # Decoder Residual Block
        self.output_embedding = nn.Sequential(
            nn.Flatten(start_dim=-2),
            nn.Linear(n_patches * embedding_dim, output_ff_dim),
            nn.Linear(output_ff_dim, h * self.loss.outputsize_multiplier),
        )

    def forward(self, windows_batch):
        
        # Parse windows_batch
        x = windows_batch['insample_y']                         # [B, L, 1]
        x_mask = windows_batch['insample_mask']                 # [B, L, 1]
        x = torch.cat((x, x_mask), dim=-1)                      # [B, L, 1] + [B, L, 1] -> [B, L, 2]

        batch_size = x.shape[0]

        # Padding
        x = x.permute(0, 2, 1)                                  # [B, L, 2] -> [B, 2, L]
        x = F.pad(x, (0, self.stride), value=0.0)               # [B, 2, L] -> [B, 2, L + stride]

        # Patching
        x = x.unfold(-1, self.input_patch_len, self.stride)     # [B, 2, L + stride] -> [B, 2, num_patches, patch_len]
        x = x.permute(0, 2, 1, 3)                               # [B, 2, num_patches, input_patch_len] -> [B, num_patches, 2, input_patch_len]
        x = x.reshape(batch_size, -1, self.input_patch_len * 2) # [B, num_patches, 2, input_patch_len] -> [B, num_patches, 2 * input_patch_len]

        # Embedding
        x = self.input_embedding(x)                             # [B, num_patches, 2 * input_patch_len] -> [B, num_patches, embedding_dim]

        # xLSTM
        x = self.xlstm(x)                                       # [B, num_patches, embedding_dim] -> [B, num_patches, embedding_dim]

        # Decoder
        x = self.output_embedding(x)                            # [B, num_patches, embedding_dim] -> [B, h * n_outputs]
        output = x.reshape(batch_size, 
                           self.h, 
                           self.loss.outputsize_multiplier)     # [B, h * n_outputs] -> [B, h, n_outputs]

        return output

In [ ]:
show_doc(TiRex)

In [ ]:
show_doc(TiRex.fit, name='TiRex.fit')

In [ ]:
show_doc(TiRex.predict, name='TiRex.predict')

In [ ]:
#| hide
# Unit tests for models
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("lightning_fabric").setLevel(logging.ERROR)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    check_model(TiRex, ["airpassengers"])

In [ ]:
#| hide
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from neuralforecast.utils import AirPassengersDF as Y_df
from neuralforecast.tsdataset import TimeSeriesDataset

## Usage Example

In [ ]:
#| eval: false
import pandas as pd
import matplotlib.pyplot as plt

from neuralforecast import NeuralForecast
from neuralforecast.models import TiRex
from neuralforecast.losses.pytorch import DistributionLoss
from neuralforecast.utils import AirPassengersPanel, AirPassengersStatic

Y_train_df = AirPassengersPanel[AirPassengersPanel.ds<AirPassengersPanel['ds'].values[-12]] # 132 train
Y_test_df = AirPassengersPanel[AirPassengersPanel.ds>=AirPassengersPanel['ds'].values[-12]].reset_index(drop=True) # 12 test

nf = NeuralForecast(
    models=[TiRex(h=12, 
                 input_size=24,
                 loss=DistributionLoss(distribution="Normal", level=[80, 90]),
                 scaler_type='robust',
                 max_steps=400,
                 )
    ],
    freq='ME'
)
nf.fit(df=Y_train_df, static_df=AirPassengersStatic)
Y_hat_df = nf.predict(futr_df=Y_test_df)

# Plots
Y_hat_df = Y_hat_df.reset_index(drop=False).drop(columns=['unique_id','ds'])
plot_df = pd.concat([Y_test_df, Y_hat_df], axis=1)
plot_df = pd.concat([Y_train_df, plot_df])

plot_df = plot_df[plot_df.unique_id=='Airline1'].drop('unique_id', axis=1)
plt.plot(plot_df['ds'], plot_df['y'], c='black', label='True')
plt.plot(plot_df['ds'], plot_df['TiRex-median'], c='blue', label='median')
plt.fill_between(x=plot_df['ds'][-12:], 
                 y1=plot_df['TiRex-lo-90'][-12:].values,
                 y2=plot_df['TiRex-hi-90'][-12:].values,
                 alpha=0.4, label='level 90')
plt.grid()
plt.plot()